# Desafio 1: Mapa de características para o Perceptron
### GBC073 - Inteligência Computacional (FACOM/UFU)

**Kaike de Morais Carvalho - 12421BCC051**
---

Desafio: kernel trick
Ao invés de usar um classificador linear, construir features não-lineares sobre o dataset (XOR, luas, círculos, espiral, esfera em alta dimensão) e usar um classificador linear em cima delas

O conjunto de dados de entrada não são linearmente separáveis no espaço original. O desafio é aumentar a dimensão de cada input (feature space) onde essas classes se tornem linearmente separáveis.

Dessa forma, o objetivo desse perceptron é realizar classificação multi-classe.

fit(X): roda uma vez só com os dados de treino. Pode inserir cálculos de média/desvio-padrão pra padronização, sortear vetores aleatórios fixos (com seed).

phi(X): aplicada tanto em treino quanto teste. Precisa:
- Aumentar a dimensão (n,d) -> (n, d') onde d é a dimensão original e d' é a dimensão nova, que deve obedecer o limite de 64.
- Ser determinística -> mesma entrada = mesma saída
- Sem NaN e Inf
- Ser rápida => vetorizada, sem loops em python puro => usar pytorch ou numpy

ex: [Xs, Xs**2] => features polinomiais -> funcionam pra XOR e círculos (dependem de termos quadráticos)


In [ ]:
pip install torch

In [ ]:
"""
desafio1_aluno.py — Desafio 1: Mapa de características para o Perceptron
GBC073 — Inteligência Computacional (FACOM/UFU)

O QUE VOCÊ FAZ: preencher a classe `Submissao` (e só ela).
O QUE O HARNESS FAZ: gera dados, divide 60/40, aplica a sua phi, treina um
Perceptron fixo e mede a acurácia. Rode:  python desafio1_aluno.py

Regras:
  * fit(X) recebe só as entradas de treino, sem rótulos. É opcional.
  * phi(X) transforma (n, d) em (n, d') com d < d' <= 64, de forma determinística.
  * Sem NaN/Inf; e rápido (10 mil pontos em menos de 2 s).
Escore: 0 = igual à identidade (baseline), 100 = igual à referência do professor,
até 125 se superar a referência. Na correção, tarefas OCULTAS da mesma família
substituem estas — não ajuste para um conjunto de dados específico.
"""


"\ndesafio1_aluno.py — Desafio 1: Mapa de características para o Perceptron\nGBC073 — Inteligência Computacional (FACOM/UFU)\n\nO QUE VOCÊ FAZ: preencher a classe `Submissao` (e só ela).\nO QUE O HARNESS FAZ: gera dados, divide 60/40, aplica a sua phi, treina um\nPerceptron fixo e mede a acurácia. Rode:  python desafio1_aluno.py\n\nRegras:\n  * fit(X) recebe só as entradas de treino, sem rótulos. É opcional.\n  * phi(X) transforma (n, d) em (n, d') com d < d' <= 64, de forma determinística.\n  * Sem NaN/Inf; e rápido (10 mil pontos em menos de 2 s).\nEscore: 0 = igual à identidade (baseline), 100 = igual à referência do professor,\naté 125 se superar a referência. Na correção, tarefas OCULTAS da mesma família\nsubstituem estas — não ajuste para um conjunto de dados específico.\n"

In [67]:
import math
import time
import torch
from torch import nn

DIM_MAX = 64
SEMENTES = (0, 1, 2)

# =============================================================================
# >>> SUA SUBMISSÃO — edite apenas esta classe <<<
# =============================================================================
class Submissao:
    DIM_MAX = DIM_MAX

    def fit(self, X: torch.Tensor) -> None:
        """Opcional: veja X de treino (sem rótulos) para padronizar, sortear projeções etc."""
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-8
        # mean(X) => retorna a média de cada linha de input dentro do tensor X
        # std(X) => retorna o desvio padrão de cada linha de input dentro do tensor X

    def phi(self, X: torch.Tensor) -> torch.Tensor:
      # (n,d) -> (n,d') => (linha,coluna) -> aumentar dimensao das colunas
        """(n, d) -> (n, d'), com d < d' <= 64. Exemplo: padroniza e acrescenta os quadrados."""
        Xs = (X - self.mu) / self.sd  # z-score


        d = Xs.shape[1] # pega a dimensão de colunas/features
        g = torch.Generator().manual_seed(0)  # garante determinismo
        w = torch.randn(d, DIM_MAX, generator=g)  # pesos
        b = torch.rand(DIM_MAX, generator=g) * 0.1  # viés pequeno
        proj = Xs @ w + b # cálculo
                # utiliza a forma de normalização usada no RRF
        return math.sqrt(2 / DIM_MAX) * torch.relu(proj) * torch.sigmoid(proj) * torch.tanh(proj) # aplica funções não-lineares pra tornar o conjunto de dados linearmente separáveis

        #return torch.cat([Xs, Xs**2], dim=1)      # d' = 2d  (troque por algo melhor!)
        # cat(X) => concatena os tensores em uma dimensão (0,1) -> linha ou coluna


In [ ]:

# =============================================================================
# Harness (não edite daqui para baixo)
# =============================================================================
"""Dados de entrada => devo estudar a característica e dimensão original de cada um => objetivo de aumentar suas dimensões pra tornar linearmente separáveis"""
def _luas(n, g):
    t = torch.rand(n // 2, generator=g) * math.pi
    X = torch.cat([torch.stack([torch.cos(t), torch.sin(t)], 1),
                   torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], 1)])
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])
    return X + 0.15 * torch.randn(n, 2, generator=g), y

def _circulos(n, g):
    t = torch.rand(n, generator=g) * 2 * math.pi
    r = torch.where(torch.arange(n) < n // 2, 1.0, 0.45)
    X = torch.stack([r * torch.cos(t), r * torch.sin(t)], 1)
    return X + 0.08 * torch.randn(n, 2, generator=g), (torch.arange(n) >= n // 2).float()

def _xor(n, g):
    X = torch.rand(n, 2, generator=g) * 2 - 1
    y = (X[:, 0] * X[:, 1] < 0).float()
    return X + 0.15 * torch.randn(n, 2, generator=g), y

def _espiral(n, g):
    t = torch.sqrt(torch.rand(n // 2, generator=g)) * 3 * math.pi
    a = torch.stack([t * torch.cos(t), t * torch.sin(t)], 1) / 10
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])
    return torch.cat([a, -a]) + 0.05 * torch.randn(n, 2, generator=g), y

def _esfera(n, g, d=10):
    X = torch.randn(n, d, generator=g)
    r2 = (X ** 2).sum(1)
    return X, (r2 > r2.median()).float()

TAREFAS = {"xor": lambda g: _xor(600, g), "duas_luas": lambda g: _luas(600, g),
           "circulos": lambda g: _circulos(600, g), "espiral": lambda g: _espiral(800, g),
           "esfera_10d": lambda g: _esfera(800, g)}


In [ ]:

# implementação do perceptron
@torch.no_grad()  # desabilita cálculo de gradiente -> útil para inferência
def perceptron_pocket(Z, y, epocas=50, eta=1.0, semente=0):
    """Regra de Rosenblatt (w <- w + eta*y*z nos erros) + pocket: guarda o melhor w."""
    Zb = torch.cat([Z, torch.ones(len(Z), 1)], 1)     # viés embutido
    yb = 2 * y - 1
    w = torch.zeros(Zb.shape[1]); melhor_w, melhor_acc = w.clone(), -1.0
    g = torch.Generator().manual_seed(semente)
    for _ in range(epocas):
        for i in torch.randperm(len(Zb), generator=g).tolist():
            if yb[i] * (Zb[i] @ w) <= 0:
                w += eta * yb[i] * Zb[i]
        acc = ((Zb @ w) * yb > 0).float().mean().item()
        if acc > melhor_acc:
            melhor_acc, melhor_w = acc, w.clone()
    return melhor_w


def acuracia_balanceada(y, yhat):
    return torch.stack([(yhat[y == c] == c).float().mean() for c in y.unique()]).mean().item()

# splita os dados de treino e teste, chama as funções implementadas, valida as regras e mede acurácia
@torch.no_grad()
def rodar(sub, gerador, semente, checar=True):
    g = torch.Generator().manual_seed(semente)  # gera tensor com vetores aleatórios fixos a partir das seeds
    X, y = gerador(g)
    idx = torch.randperm(len(X), generator=g); ntr = int(0.6 * len(X))
    Xtr, ytr, Xte, yte = X[idx[:ntr]], y[idx[:ntr]], X[idx[ntr:]], y[idx[ntr:]]

    torch.manual_seed(semente)
    sub.fit(Xtr)                                       # nunca recebe ytr
    t0 = time.perf_counter(); Ztr = sub.phi(Xtr); dt = time.perf_counter() - t0
    Zte = sub.phi(Xte)
    if checar:                                         # regras do desafio
        d, dl = Xtr.shape[1], Ztr.shape[1]
        assert Ztr.ndim == 2 and Zte.shape[1] == dl, "phi deve devolver (n, d')"
        assert d < dl <= DIM_MAX, f"exige d < d' <= {DIM_MAX}; recebi d={d}, d'={dl}"
        assert torch.isfinite(Ztr).all() and torch.isfinite(Zte).all(), "NaN/Inf na saída de phi"
        assert torch.allclose(sub.phi(Xtr[:20]), Ztr[:20]), "phi não é determinística"
        assert dt * (10_000 / len(Xtr)) < 2.0, "phi lenta demais (limite: 10^4 pontos em 2 s)"

    w = perceptron_pocket(Ztr, ytr, semente=semente)
    yhat = (torch.cat([Zte, torch.ones(len(Zte), 1)], 1) @ w > 0).float()
    return acuracia_balanceada(yte, yhat)


In [68]:
class _Identidade:                       # baseline (viola d < d', mas é só o ponto zero da escala)
    def fit(self, X): pass
    def phi(self, X): return X

class _RFF:                              # referência: random Fourier features, d' = 64
    def fit(self, X):
        g = torch.Generator().manual_seed(0)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-8
        Xs = (X - self.mu) / self.sd
        d2 = torch.cdist(Xs[:300], Xs[:300]) ** 2
        sigma = math.sqrt(d2[d2 > 0].median().item() / 2)
        self.W = torch.randn(X.shape[1], DIM_MAX, generator=g) / sigma
        self.b = torch.rand(DIM_MAX, generator=g) * 2 * math.pi
    def phi(self, X):
        return math.sqrt(2 / DIM_MAX) * torch.cos(((X - self.mu) / self.sd) @ self.W + self.b)


def _mediana(cls, gerador, checar=True):
    vals = [rodar(cls(), gerador, sem, checar) for sem in SEMENTES]
    return float(torch.tensor(vals).median())


def avaliar():
    print(f"{'tarefa':<12}{'baseline':>10}{'referência':>12}{'você':>8}{'s_t':>7}")
    s = []
    for nome, gen in TAREFAS.items():
        b = _mediana(_Identidade, gen, checar=False)
        r = max(_mediana(_RFF, gen, checar=False), b + 1e-3)
        try:
            m = _mediana(Submissao, gen); erro = ""
        except AssertionError as e:
            m, erro = b, f"   <- {e}"
        st = min(max((m - b) / (r - b), 0.0), 1.25); s.append(st)
        print(f"{nome:<12}{b:>10.3f}{r:>12.3f}{m:>8.3f}{st:>7.2f}{erro}")
    S = 100 * (0.7 * sum(s) / len(s) + 0.3 * min(s))
    print(f"\nESCORE S = {S:.1f}   (0 = baseline, 100 = referência, até 125 com bônus)")
    return S


if __name__ == "__main__":
    avaliar()

tarefa        baseline  referência    você    s_t
xor              0.606       0.876   0.896   1.08
duas_luas        0.892       0.983   0.984   1.01
circulos         0.644       1.000   1.000   1.00
espiral          0.633       0.912   0.787   0.55
esfera_10d       0.508       0.872   0.871   1.00

ESCORE S = 81.6   (0 = baseline, 100 = referência, até 125 com bônus)
